# 05 — Build Supplementary Files S1 and S2

Run this notebook after the benchmark and robustness notebooks. It creates two Excel workbooks and one ZIP bundle under `outputs/supplementary/`. It reads generated tables when available and uses manuscript-aligned final summary values for the compact summary sheets.


In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
OUT = ROOT / 'outputs'
SUPP = OUT / 'supplementary'
SUPP.mkdir(parents=True, exist_ok=True)
S1 = SUPP / 'Supplementary_File_S1_AllModels_HorizonWise_Metrics.xlsx'
S2 = SUPP / 'Supplementary_File_S2_10Seed_Robustness_Analyses.xlsx'
ZIP = SUPP / 'LASH_Supplementary_Files_S1_S2.zip'
protocol = json.loads((ROOT/'configs'/'final_protocol.json').read_text(encoding='utf-8'))
SEEDS = protocol['final_seeds']


In [ ]:
def newest(pattern):
    hits = list(OUT.rglob(pattern))
    return max(hits, key=lambda p: p.stat().st_mtime) if hits else None

def read_table(path):
    if path is None: return None
    if path.suffix.lower() == '.csv': return pd.read_csv(path)
    if path.suffix.lower() in ('.xlsx','.xls'): return pd.read_excel(path)
    return None

def add_df(writer, df, name):
    if df is not None and len(df):
        df.to_excel(writer, sheet_name=name[:31], index=False)

def style_book(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        ws.freeze_panes = 'A2'
        ws.sheet_view.showGridLines = False
        for c in ws[1]:
            c.font = Font(name='Arial', size=10, bold=True, color='FFFFFF')
            c.fill = PatternFill('solid', fgColor='1F4E78')
            c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        for col in range(1, ws.max_column+1):
            letter = get_column_letter(col)
            maxlen = max([len(str(ws.cell(r,col).value or '')) for r in range(1,min(ws.max_row,250)+1)] or [8])
            ws.column_dimensions[letter].width = min(max(maxlen+2,10),42)
        for row in ws.iter_rows(min_row=2):
            for c in row:
                c.font = Font(name='Arial', size=9)
    wb.save(path)


In [ ]:
# Manuscript-aligned compact benchmark summary (composite score mean ± SD).
table9 = pd.DataFrame([
['LASH',5.3708,.0137,5.0464,.0094,2.3776,.0000,4.6052,.0000],
['Ridge',5.5004,np.nan,5.0739,np.nan,2.3776,np.nan,4.6052,np.nan],
['LightGBM',5.4149,.0018,5.6798,.0046,2.5059,.0009,7.0699,.0095],
['CatBoost',5.4746,.0030,5.5224,.0060,2.5100,.0023,7.2685,.0331],
['XGBoost',5.5863,.0054,5.6674,.0068,2.5133,.0008,7.0605,.0062],
['GBM',5.8679,.0040,5.9300,.0100,2.5305,.0021,7.4518,.0232],
['RF',5.7384,.0029,5.9510,.0042,2.6143,.0000,7.2282,.0000],
['MLP',5.7009,.0284,5.3872,.0292,2.6143,.0000,7.7082,.1571],
['LSTM',6.3466,.0851,6.0177,.0916,2.5670,.0441,7.9545,.2464],
['GRU',6.2363,.1555,6.1371,.0936,2.5968,.0260,7.1692,.1370],
['CNN-LSTM',6.2724,.0598,6.2098,.1126,2.7209,.1365,8.6878,.4749],
['TCN',6.1160,.0715,5.7032,.0534,2.6193,.0214,9.1942,.5002],
['Transformer',6.5694,.0963,6.1152,.1192,2.6143,.0000,7.8133,.5032],
['Anchor',7.6058,np.nan,7.2760,np.nan,2.6143,np.nan,7.2282,np.nan],
['Seasonal24',14.5937,np.nan,10.7033,np.nan,5.7984,np.nan,4.5827,np.nan],
['Seasonal168',12.7469,np.nan,10.4841,np.nan,4.1068,np.nan,10.0809,np.nan]],
columns=['Model','Cluster1_Mean','Cluster1_SD','Cluster2_Mean','Cluster2_SD','BDG_Edu_Mean','BDG_Edu_SD','BDG_Dorm_Mean','BDG_Dorm_SD'])

seed_audit = pd.DataFrame({
 'Model':['RF','GBM','XGBoost','LightGBM','CatBoost','MLP','LSTM','GRU','CNN-LSTM','TCN','Transformer','LASH','Ridge','Anchor','Seasonal24','Seasonal168'],
 'Type':['Stochastic']*12+['Deterministic']*4,
 'Expected_Evaluations':[10]*12+[1]*4,
 'Seeds_or_Evaluation':[', '.join(map(str,SEEDS))]*12+['Single evaluation']*4})

readme1 = pd.DataFrame([
['Purpose','Complete horizon-specific MAPE, CVRMSE, and NMAE evidence for the 16-model benchmark.'],
['Final stochastic seeds',', '.join(map(str,SEEDS))],
['Deterministic references','Seasonal-24, Seasonal-168, Anchor, Ridge; evaluated once.'],
['Selection boundary','No test observation is used for preprocessing, HPO, routing, or architecture selection.']], columns=['Item','Description'])

horizon_files = list(OUT.rglob('*horizon*.csv'))
seed_files = list(OUT.rglob('*seed*metric*.csv'))
with pd.ExcelWriter(S1, engine='openpyxl') as w:
    add_df(w, readme1, '00_README')
    add_df(w, table9, '01_Final_Composite')
    add_df(w, seed_audit, '02_Seed_Audit')
    for i,p in enumerate(horizon_files[:20],1): add_df(w, pd.read_csv(p), f'Horizon_{i:02d}')
    for i,p in enumerate(seed_files[:20],1): add_df(w, pd.read_csv(p), f'Seed_{i:02d}')
style_book(S1)
print('Created:', S1)


In [ ]:
focal = newest('focal_effect_sizes_all_blocks.csv') or newest('focal_effect_sizes_and_hierarchical_inference.csv')
all168 = newest('all_baseline_primary168h_hierarchical.csv')
structural = newest('all_four_structural_feature_metrics_10seed.csv')
annual = newest('cluster_yoy_metrics_10seed.csv')
storage = newest('common_constrained_storage_scheduling_summary.csv')
reduced = newest('selected_reduced_10seed_metrics_all_datasets.csv')
retained = newest('retained_component_incremental_value.csv')

focal168 = pd.DataFrame([
['Cluster 1','LightGBM',-0.0450486726,0.9658213477,0.1327734453,0.0823125087,'Favorable direction; primary 168-h Holm support not established'],
['Cluster 1','Ridge',-0.1147460297,2.4238788810,0.00239952,0.4210030731,'Supported and above 1% practical reference'],
['Cluster 2','Ridge',-0.0262762922,0.5851764241,0.00239952,0.2465093720,'Supported but below 1% practical reference'],
['BDG_Edu','Ridge',0.0,0.0,1.0,0.0,'Identical'],
['BDG_Dorm','Seasonal24',-0.0301873141,0.7717118903,1.0,0.0158480022,'Unsupported'],
['BDG_Dorm','Ridge',0.0,0.0,1.0,0.0,'Identical']],
columns=['Dataset','Comparator','Delta_NMAE_pp','Relative_Improvement_pct','Holm_p','Hedges_g_block','Interpretation'])

storage_medium = pd.DataFrame([
['Cluster 1','LASH',7.423,3.864,.592],['Cluster 1','LightGBM',7.363,3.993,.585],
['Cluster 2','LASH',4.973,2.999,.449],['Cluster 2','Ridge',4.884,3.051,.441],
['BDG_Edu','LASH',5.704,3.431,.661],['BDG_Edu','Ridge',5.704,3.431,.661],
['BDG_Dorm','LASH',2.103,4.949,.221],['BDG_Dorm','Seasonal24',2.121,3.282,.259]],
columns=['Dataset','Model','Peak_Reduction_Mean_pct','Peak_Reduction_SD_pct','Oracle_Capture_Mean'])

audit = pd.DataFrame([
['Final stochastic seeds',', '.join(map(str,SEEDS))],['Bootstrap replicates',5000],
['Dependence blocks','24, 72, 168, 336 h'],['Primary block','168 h'],
['Practical magnitude reference','1.0% relative NMAE'],['Primary focal family','6 contrasts per block scale'],
['Storage round-trip efficiency',0.90],['Test used for selection','No']], columns=['Audit_Item','Final_Setting'])

readme2 = pd.DataFrame([
['Purpose','Dependence-aware inference, architecture robustness, annual-component, storage scheduling, and protocol audit.'],
['Inference','Hierarchical circular-block bootstrap; 5000 replicates; 168 h primary.'],
['Multiplicity','Holm adjustment across six predefined focal contrasts at each block scale.'],
['Practical magnitude','1.0% relative NMAE interpreted separately from statistical support.'],
['Storage scope','Offline constrained scheduling sensitivity, not realized deployment benefit.']], columns=['Item','Description'])

with pd.ExcelWriter(S2, engine='openpyxl') as w:
    add_df(w, readme2, '00_README')
    add_df(w, read_table(focal), '01_Focal_AllBlocks')
    add_df(w, focal168, '02_Focal_Primary168')
    add_df(w, read_table(all168), '03_Secondary_All168')
    add_df(w, read_table(structural), '04_Structural_Raw')
    add_df(w, read_table(reduced), '05_Reduced_Architecture')
    add_df(w, read_table(retained), '06_Retained_Components')
    add_df(w, read_table(annual), '07_Annual_Raw')
    add_df(w, read_table(storage), '08_Storage_AllScenarios')
    add_df(w, storage_medium, '09_Storage_Medium')
    add_df(w, audit, '10_Protocol_Audit')
style_book(S2)
print('Created:', S2)

with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(S1, S1.name); z.write(S2, S2.name)
print('Created:', ZIP)
